# Sprint 4 - Graph C GCN Runner

**Runner-only notebook. It contains no model, preprocessing, evaluation, or plotting logic.**
All scientific code lives in repository files under `src/`, `scripts/`, and `configs/`.

Slice 5B execution plan: `docs/exec-plans/completed/004-sprint4-gcn-baseline.md`  
Runner boundary: `colab/README.md`

---
**Before starting:**
- [ ] Colab runtime is set to GPU.
- [ ] Drive contains `crispr_gnn_offtarget/data/processed/graphs/sprint3/`.
- [ ] The branch or commit includes Slice 5A Graph C code and `configs/experiments/gcn_graph_c.yaml`.
- [ ] Graph A Slice 4C is already validated.

Graph C changes both topology and target semantics/context representation relative to Graph A. Do not describe it as a topology-only experiment.

## ADIM 1 - Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ADIM 2 - Repo Clone and Checkout

In [ ]:
%%bash
pip install uv --quiet
git clone https://github.com/YasinEkici/crispr-gnn-offtarget.git crispr-gnn-offtarget
cd crispr-gnn-offtarget
git checkout sprint4/gcn-baseline
echo "=== Commit SHA ==="
git rev-parse HEAD

## ADIM 3 - Dependency Sync and Version Check

If `cuda_available: False`, the run can still execute on CPU but should be treated as provisional, not as the intended full GPU run.

In [ ]:
%%bash
cd crispr-gnn-offtarget
uv sync
echo "=== Version Check ==="
uv run python -c "
import torch, torch_geometric
print('torch         :', torch.__version__)
print('pyg           :', torch_geometric.__version__)
print('cuda_available:', torch.cuda.is_available())
print('cuda_version  :', torch.version.cuda)
print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
"

## ADIM 4 - Run ID and Resolved Config Preparation

Update the date. If running more than once on the same day, add `_v2`, `_v3`, etc. The base config remains unchanged.

In [ ]:
RUN_ID = "sprint4_graph_c_gcn_seed42_20260601"  # update if needed
print("Run ID:", RUN_ID)

In [ ]:
from pathlib import Path

import torch
import yaml

base_config_path = Path("crispr-gnn-offtarget/configs/experiments/gcn_graph_c.yaml")
run_dir = Path(f"crispr-gnn-offtarget/outputs/sprint4/graph_c/{RUN_ID}")
run_dir.mkdir(parents=True, exist_ok=True)
resolved_config_path = run_dir / "resolved_config.yaml"

with open(base_config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
config["run_id"] = RUN_ID
config.setdefault("training", {})["device"] = device

assert config["task"] == "sprint4_gcn_graph_c"
assert config["graph"]["schema"] == "graph_c_context_observation"
assert config["data"]["split_id"] == "sprint2_main_seed42"
assert config["data"]["label_scheme"] == "scheme_a"
assert config["graph"]["visibility_policy"] == "strict_inductive_primary"
assert config["training"]["loss"] == "weighted_bce"
assert config["evaluation"]["protocol"] == "headline_guide_level"

with open(resolved_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(f"Base config unchanged: {base_config_path}")
print(f"Resolved run config:  {resolved_config_path}")
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: CUDA unavailable. This run is provisional, not the intended full GPU run.")

print("\nResolved fields:")
print(f"task: {config['task']}")
print(f"run_id: {config['run_id']}")
print(f"device: {config['training']['device']}")
print(f"schema: {config['graph']['schema']}")
print(f"target representation: {config['model']['target_node_representation']}")

## ADIM 5 - Copy Sprint 3 Artifacts From Drive

Drive must contain `crispr_gnn_offtarget/data/processed/graphs/sprint3/`.

In [ ]:
%%bash
DRIVE_SPRINT3="/content/drive/MyDrive/crispr_gnn_offtarget/data/processed/graphs/sprint3"
LOCAL_GRAPHS="crispr-gnn-offtarget/data/processed/graphs"

mkdir -p "${LOCAL_GRAPHS}"
cp -r "${DRIVE_SPRINT3}" "${LOCAL_GRAPHS}/"

echo "=== Copied graph artifact folders ==="
ls "${LOCAL_GRAPHS}/sprint3/"

## ADIM 6 - Provenance Gate (Required)

Do not start ADIM 7 if this step fails. This gate validates the full Sprint 3 artifact set and explicitly asserts Graph C schema counts.

In [ ]:
import json
import os
import subprocess

provenance_path = f"crispr-gnn-offtarget/outputs/sprint4/graph_c/{RUN_ID}/graph_artifact_provenance.json"
os.makedirs(f"crispr-gnn-offtarget/outputs/sprint4/graph_c/{RUN_ID}", exist_ok=True)

result = subprocess.run(
    [
        "uv", "run", "python", "scripts/validate_graph_artifacts.py",
        "--artifact-dir", "data/processed/graphs/sprint3",
        "--approved-source", "drive_sprint3_handoff",
        "--output", f"outputs/sprint4/graph_c/{RUN_ID}/graph_artifact_provenance.json",
    ],
    cwd="crispr-gnn-offtarget",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
    raise RuntimeError("Provenance gate failed. Do not start ADIM 7.")

with open(provenance_path, encoding="utf-8") as f:
    prov = json.load(f)

graph_c = prov["schemas"]["graph_c_context_observation"]
assert graph_c["graph_name"] == "graph_c_context_observation"
assert graph_c["nodes"]["sgRNA"] == 150
assert graph_c["nodes"]["target_observation"] == 11446
assert graph_c["relations"]["candidate_pair"] == 11446
assert graph_c["relations"]["context_similar_to"] == 91754
assert graph_c["candidate_pair_edges"] == 11446
assert graph_c["split_id"] == "sprint2_main_seed42"
assert graph_c["label_scheme"] == "scheme_a"
assert graph_c["visibility_policy"] == "strict_inductive_primary"

print("\n=== Schema Summary ===")
for schema, info in prov["schemas"].items():
    print(f"{schema[:35]:35} | edges: {info['candidate_pair_edges']} | split: {info['split_id']}")
print("\nGraph C counts validated:", graph_c["nodes"], graph_c["relations"])
print("Provenance gate passed. Training may start.")

## ADIM 7 - Graph C Training

This command writes Graph C outputs under `outputs/sprint4/graph_c/`. Test metrics are interpretation-only; do not change any model, feature, epoch, checkpoint, threshold, or reporting choice based on test diagnostics.

In [ ]:
import subprocess

result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/train.py",
        "--config",
        f"outputs/sprint4/graph_c/{RUN_ID}/resolved_config.yaml",
    ],
    cwd="crispr-gnn-offtarget",
    text=True,
)
if result.returncode != 0:
    raise RuntimeError("Graph C training command failed")

## ADIM 8 - Copy Artifacts Back To Drive

In [ ]:
import os
import shutil

DRIVE_OUT = f"/content/drive/MyDrive/crispr_gnn_offtarget/returned_outputs/{RUN_ID}"
os.makedirs(DRIVE_OUT, exist_ok=True)

base = "crispr-gnn-offtarget"

copies = [
    (f"{base}/outputs/sprint4/graph_c/{RUN_ID}", f"{DRIVE_OUT}/{RUN_ID}"),
    (f"{base}/outputs/sprint4/graph_c/gcn_graph_c_results.csv", f"{DRIVE_OUT}/gcn_graph_c_results.csv"),
    (f"{base}/outputs/sprint4/graph_c/gcn_graph_c_report.md", f"{DRIVE_OUT}/gcn_graph_c_report.md"),
    (f"{base}/outputs/sprint4/graph_c/diagnostics", f"{DRIVE_OUT}/diagnostics"),
    (f"{base}/outputs/sprint4/graph_c/figures", f"{DRIVE_OUT}/figures"),
]

for src, dst in copies:
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy2(src, dst)
    else:
        print(f"WARNING: Missing artifact: {src}")
        continue
    print(f"OK: {src} -> {dst}")

## ADIM 9 - Returned Artifact Verification Checklist

In [ ]:
import os

DRIVE_OUT = f"/content/drive/MyDrive/crispr_gnn_offtarget/returned_outputs/{RUN_ID}"

required = [
    f"{DRIVE_OUT}/{RUN_ID}/graph_artifact_provenance.json",
    f"{DRIVE_OUT}/{RUN_ID}/resolved_config.yaml",
    f"{DRIVE_OUT}/{RUN_ID}/runtime.json",
    f"{DRIVE_OUT}/{RUN_ID}/model.pt",
    f"{DRIVE_OUT}/{RUN_ID}/training_history.csv",
    f"{DRIVE_OUT}/gcn_graph_c_results.csv",
    f"{DRIVE_OUT}/gcn_graph_c_report.md",
]

required_figures = [
    "gcn_graph_c_graph_schema_auprc_comparison.png",
    "gcn_graph_c_pr_curves.png",
    "gcn_graph_c_roc_curves.png",
    "gcn_graph_c_training_curves.png",
    "gcn_graph_c_score_distributions.png",
    "gcn_graph_c_confusion_matrices.png",
    "gcn_graph_c_decile_lift.png",
    "gcn_graph_c_per_genome_metrics.png",
    "gcn_graph_c_view_sanity_example.png",
]

required_diagnostics = [
    "gcn_graph_c_predictions.csv",
    "gcn_graph_c_training_history.csv",
    "gcn_graph_c_score_direction.csv",
    "gcn_graph_c_fixed_threshold_metrics.csv",
    "gcn_graph_c_score_deciles.csv",
    "gcn_graph_c_per_genome_metrics.csv",
    "gcn_graph_c_test_per_guide_metrics.csv",
]

all_ok = True
print("=== Required artifacts ===")
for path in required:
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'MISSING'} {os.path.basename(path)}")
    if not exists:
        all_ok = False

print("\n=== Required figures (figures/) ===")
for fig in required_figures:
    path = f"{DRIVE_OUT}/figures/{fig}"
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'MISSING'} {fig}")
    if not exists:
        all_ok = False

print("\n=== Required diagnostic tables (diagnostics/) ===")
for diag in required_diagnostics:
    path = f"{DRIVE_OUT}/diagnostics/{diag}"
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'MISSING'} {diag}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All required Graph C artifacts are present. Ready for returned-artifact validation.")
else:
    print("Some Graph C artifacts are missing. Do not claim Slice 5B/5C completion.")